<a href="https://colab.research.google.com/github/Dwayne-tech/DML/blob/main/Anxiety_Attack_Risk_Assessment_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Installing the required packages
!pip install streamlit           # Streamlit is a framework to create web apps for machine learning models
!pip install pyngrok             # Pyngrok is used to expose the local Streamlit app to the web via a public URL
!pip install streamlit-option-menu # Option menu for creating navigation in the app

In [ ]:
%%writefile app.py
# Import necessary libraries
import streamlit as st                     # Streamlit to build the app's interface
import pandas as pd                        # Pandas for data handling
import numpy as np                         # Numpy for numerical computations
from sklearn.model_selection import train_test_split  # For splitting data into training and test sets
from sklearn.preprocessing import StandardScaler      # For scaling numerical features
from sklearn.ensemble import RandomForestClassifier   # Random Forest model for classification

# Function to load the anxiety attack dataset
def load_data():
    # Load the dataset (ensure path is correct)
    data = pd.read_csv('/content/anxiety_attack_dataset.csv')

    # Column names
    column_names = ["ID", "Age", "Gender", "Occupation", "Sleeping Hours",
                    "Physical Activity (hrs/week)", "Caffeine Intake (mg/day)",
                    "Alcohol Consumption (drinks/week)", "Smoking",
                    "Family History of Anxiety", "Stress Level (1-10)",
                    "Heart Rate (bpm during attack)", "Breathing Rate (breaths/min)",
                    "Sweating Level (1-5)", "Dizziness", "Medication",
                    "Therapy Sessions (per month)", "Recent Major Life Event",
                    "Diet Quality (1-10)", "Severity of Anxiety Attack (1-10)"]
    data.columns = column_names  # Assign correct column names

    # Replace '?' with NaN
    data = data.replace('?', np.nan)

    # Convert all columns to numeric where possible
    for column in data.columns:
        data[column] = pd.to_numeric(data[column], errors='coerce')

    # Fill missing values with median for numeric columns
    for column in data.select_dtypes(include=['float64', 'int64']).columns:
        data[column] = data[column].fillna(data[column].median())

    return data

# Function to train the Random Forest model
def train_model(data):
    # Selecting features (X) and target (y)
    X = data.drop(['ID', 'Severity of Anxiety Attack (1-10)'], axis=1)  # Dropping non-predictive columns like ID
    y = data['Severity of Anxiety Attack (1-10)'].apply(lambda x: 1 if x > 5 else 0)  # Binary classification: 1 for severe (rating > 5), else 0

    # Split the data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Scaling the feature variables
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    # Training a Random Forest classifier
    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train_scaled, y_train)

    return model, scaler  # Return the trained model and the scaler for later use

# Main function to define the Streamlit app layout
def main():
    # Set the app title and introductory text
    st.title("Anxiety Attack Risk Assessment")
    st.write("Your interactive guide to assess risk factors for anxiety attacks")

    # Load and train the model
    @st.cache_resource
    def load_trained_model():
        data = load_data()
        model, scaler = train_model(data)
        return model, scaler, data

    # Load the model, scaler, and data
    model, scaler, data = load_trained_model()

    # Create two tabs: Chat Assistant and Risk Assessment
    tab1, tab2 = st.tabs(["Chat Assistant", "Risk Assessment"])

    with tab1:
        st.subheader("Ask me about anxiety attacks!")
        user_query = st.selectbox("Choose your question:",
                                  ["What are anxiety attacks?",
                                   "What are the symptoms?",
                                   "How can I reduce anxiety?"])
        if st.button("Get Answer"):
            if user_query == "What are anxiety attacks?":
                st.write("Anxiety attacks are sudden episodes of intense fear or anxiety.")
            elif user_query == "What are the symptoms?":
                st.write("Symptoms include rapid heartbeat, difficulty breathing, sweating, and dizziness.")
            else:
                st.write("To reduce anxiety, try relaxation techniques, therapy, and maintaining a balanced lifestyle.")

    with tab2:
        st.subheader("Anxiety Attack Risk Assessment")
        st.write("Enter your health information to assess the risk of an anxiety attack:")

        # Input fields for health details
        age = st.number_input("Age", 18, 100, 30)
        gender = st.selectbox("Gender", ["Male", "Female"])
        sleeping_hours = st.number_input("Sleeping Hours", 0, 24, 7)
        physical_activity = st.number_input("Physical Activity (hrs/week)", 0, 20, 3)
        caffeine_intake = st.number_input("Caffeine Intake (mg/day)", 0, 1000, 200)
        alcohol_consumption = st.number_input("Alcohol Consumption (drinks/week)", 0, 20, 0)
        smoking = st.selectbox("Smoking", ["Yes", "No"])
        family_history = st.selectbox("Family History of Anxiety", ["Yes", "No"])
        stress_level = st.number_input("Stress Level (1-10)", 1, 10, 5)
        therapy_sessions = st.number_input("Therapy Sessions (per month)", 0, 10, 0)

        # Assess risk on button click
        if st.button("Assess Risk"):
            input_data = np.array([[age, 1 if gender == "Male" else 0, sleeping_hours, physical_activity,
                                    caffeine_intake, alcohol_consumption, 1 if smoking == "Yes" else 0,
                                    1 if family_history == "Yes" else 0, stress_level, therapy_sessions]])

            # Scale the input data
            scaled_data = scaler.transform(input_data)
            prediction = model.predict(scaled_data)  # Get the model's prediction

            st.write("The model predicts: " + ("High Anxiety Risk" if prediction == 1 else "Low Anxiety Risk"))

if __name__ == "__main__":
    main()


Writing app.py


In [ ]:
from pyngrok import ngrok

# Install Ngrok authtoken
!ngrok authtoken 2nsOJTOt5ump7Vz8YhWmAVMCo9U_3FV6cMBUrkg8rGm3RyLQn

# Run Streamlit app
!streamlit run app.py &>/dev/null&

# Create an Ngrok tunnel for port 8501 (Streamlit default port)
public_url = ngrok.connect(8501, "http")

# Print the public URL to access the app
print(public_url)


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
NgrokTunnel: "https://2770-34-147-2-87.ngrok-free.app" -> "http://localhost:8501"
